# Lab 06 — Analyzing an Annotation Pilot

In this notebook, we explore the results of a small annotation pilot conducted in Label Studio.

We want to understand what happened during annotation, identify potentially problematic examples, and decide how to improve our annotation task.

We will look at annotation progress, time spent, and differences from existing reference labels. These are useful signals for investigating the task, but they do not necessarily tell us whether an annotator is good or bad.

Formal annotation quality and agreement will be covered in Lab 07.

### Load the pilot

In [ ]:

from utils.analysis import load_pilot

ANNOTATIONS_FILE = "pilot-result.json"
REFERENCE_FILE = "export/pilot_reference.csv"

# Use a mapping only if Label Studio and the dataset use different labels
LABEL_MAPPING = {}

items, annotations = load_pilot(
    ANNOTATIONS_FILE,
    REFERENCE_FILE,
    label_mapping=LABEL_MAPPING,
)


## 1. What happened in our pilot?

Let's start with an overview of the annotations we collected.

How many examples were annotated? How many people participated? Did all examples receive the requested number of annotations?

An example with one annotation is not necessarily complete if our task requires two independent judgments.

In [ ]:
from IPython.display import display
from utils.analysis import pilot_summary, plot_annotation_coverage

NUM_RATINGS_PER_EXAMPLE = 2

display(pilot_summary(items, annotations, NUM_RATINGS_PER_EXAMPLE))
fig, ax = plot_annotation_coverage(items, NUM_RATINGS_PER_EXAMPLE)


## 2. Do annotators improve over time?

Annotators may become more familiar with the task as they work through examples. They might make decisions faster or interpret the labels more consistently.

Let's examine how annotation time changes as each worker completes more examples.

We use the submission timestamps to reconstruct the order of annotations for each worker.

Keep in mind that changes in annotation time may also reflect differences in example difficulty, interruptions, or fatigue.

In [ ]:
from utils.analysis import plot_annotator_progress

fig, ax = plot_annotator_progress(annotations)

## 3. Where might there be problems?

### 3.1 Annotation time

Some examples may take longer to annotate than others. This can indicate difficult decisions, unclear instructions, or missing information.

However, a long annotation time can also mean that someone was interrupted or left the task open.

Let's look at the distribution of annotation times and inspect the slowest examples.

In [ ]:

from utils.analysis import plot_annotation_times

plot_annotation_times(annotations)

# display the elements that took the most time
annotations.sort_values("lead_time", ascending=False)[
    ["example_id", "text", "label", "lead_time"]
].head(5)


### 3.2 Differences from reference labels

For our banking example, we already have labels from the original dataset. We can compare them with the labels collected in the pilot.

A difference does not automatically mean that the annotator made a mistake. It may reveal unclear category definitions, an ambiguous message, or a problem with the original label.

Let's inspect some examples where the labels differ.

In [ ]:
mismatches = annotations[annotations["reference_match"] == False]

mismatches[
    ["example_id", "text", "reference_label", "label", "annotator_id"]
].head(10)

### 3.3 Are some labels more difficult than others?

Some categories may be harder to distinguish than others. Annotators might spend more time on certain labels or select labels that differ more often from our existing reference data.

Let's compare annotation time and reference-label matches across categories.

Keep in mind that we are working with a small pilot. These differences may help us identify cases worth investigating, but they are not definitive measures of difficulty or correctness.

In [ ]:
by_label = annotations.groupby("reference_label").agg(
    n=("label", "count"),
    median_time=("lead_time", "median"),
    reference_match=("reference_match", "mean")
)

display(by_label.round(2))

In [ ]:
by_label["median_time"].sort_values().plot.barh(
    xlabel="Median annotation time (seconds)",
    ylabel="Reference label",
    figsize=(8, 4)
)

### 3.4 Does message length matter?

Longer messages may require more reading and therefore take longer to annotate. But short messages can also be difficult when they lack context or are ambiguous.

Let's examine whether message length is associated with annotation time.

In [ ]:
annotations["message_length"] = annotations["text"].str.len()

annotations.plot.scatter(
    x="message_length",
    y="lead_time",
    figsize=(7, 4),
    xlabel="Message length (characters)",
    ylabel="Annotation time (seconds)"
)

## 4. Take a closer look

The summaries help us identify potential problems, but they do not explain why they occurred.

Choose an example that seems interesting. Look at the original message, the labels assigned by different annotators, and the time spent.

Does the problem seem related to the example itself, the category definitions, or the instructions?

You can change the example ID below to inspect another item.

In [ ]:
EXAMPLE_ID = "train:8864"

example = annotations[annotations["example_id"] == EXAMPLE_ID]

# display the full message
print("Customer message:")
print(example["text"].iloc[0])

# annotations from all the workers
display(example[
    ["annotator_id", "label", "reference_label", "lead_time"]
])

## 5. Estimating annotation costs

Human effort costs money. When planning an annotation task, we need to consider not only how many examples we want to annotate, but also how much time and money this will require.

Label Studio records the time spent on each annotation (`lead_time`). We can use this information to estimate annotation costs, but there are some limitations:

- The recorded time may include interruptions or periods when the annotator was inactive.
- It does not include other work, such as preparing instructions, training annotators, or reviewing results.
- Annotation times from a small pilot may not represent a larger task.

Our estimates are therefore approximate, but they can help us plan annotation budgets.

### 5.1 How much did our pilot cost?

Let's calculate the total recorded annotation time and estimate the cost of our pilot using an assumed hourly rate.

We can also calculate the average cost per annotation.

In [ ]:
HOURLY_RATE = 18  # EUR per hour — hypothetical rate

costs = annotations.dropna(subset=["lead_time", "label"]).copy()
costs["cost"] = costs["lead_time"] / 3600 * HOURLY_RATE

print(f"Total annotation time: {costs['lead_time'].sum() / 60:.1f} minutes")
print(f"Estimated pilot cost: €{costs['cost'].sum():.2f}")
print(f"Average cost per annotation: €{costs['cost'].mean():.2f}")

### 5.2 Scaling up 

We can use the average annotation time from our pilot to estimate the cost of annotating a larger dataset.

The estimated cost depends on the number of examples, how many independent annotations we collect per example, and the hourly rate.

How would the cost change if we required three annotations instead of two? How reliable is an estimate based on our small pilot?

In [ ]:
NUM_EXAMPLES = 1000
RATINGS_PER_EXAMPLE = 2

estimated_cost = (
    NUM_EXAMPLES
    * RATINGS_PER_EXAMPLE
    * costs["cost"].mean()
)

print(f"Estimated annotation cost: €{estimated_cost:,.2f}")

**Discussion**

- How much would it cost to annotate your full dataset?
- How would the cost change if each example required three annotations instead of two?
- How could we reduce annotation costs without compromising the usefulness of the data?

## 6. What should we improve?

Based on your pilot results, identify two or three examples or observations that deserve attention.

If your annotation task collected **feedback from annotators**, consider this information as well. You can inspect their comments to understand what was difficult or confusing. If feedback was categorized, you can also examine which problems were reported most frequently.

For each observation, consider:

- **Observation:** What happened?
- **Possible cause:** What might explain it?
- **Revision:** What would you change in the annotation instructions, labels, examples, or interface?

Which of these changes would you test in a second pilot?

**Discussion:** Can we conclude from this pilot that our annotations are reliable? What additional analysis would we need?

We will explore these questions in Lab 07.

In [ ]:
# If your task collected qualitative feedback, you can inspect it here.
# This assumes the feedback fields have been extracted into the annotations DataFrame.

# List annotator comments
# display(annotations[["example_id", "feedback_text"]].dropna())

# Count reported issues by category
# display(annotations["feedback_category"].value_counts())